In [1]:
import requests
import pandas as pd

# Lista de monedas y sus IDs en CoinGecko
coins = {
    'pepe': 'pepe',
    'doge': 'dogecoin',
    'shiba': 'shiba-inu'
}

# Lista para almacenar los DataFrames de cada moneda
all_prices = []

# Iterar por cada moneda
for name, coingecko_id in coins.items():
    print(f"Descargando datos de {name.upper()}...")

    url = f'https://api.coingecko.com/api/v3/coins/{coingecko_id}/market_chart'
    params = {
        'vs_currency': 'usd',
        'days': '365'
    }

    response = requests.get(url, params=params)
    data = response.json()

    # Crear DataFrame
    prices = data['prices']
    df = pd.DataFrame(prices, columns=['timestamp', 'price'])

    # Convertir timestamp a fecha (sin hora)
    df['date'] = pd.to_datetime(df['timestamp'], unit='ms').dt.date

    # Agrupar por fecha y calcular la media
    daily_df = df.groupby('date', as_index=False).mean()

    # Añadir columna con el nombre de la moneda
    daily_df['coin'] = name

    # Añadir a la lista
    all_prices.append(daily_df)

# Concatenar todos los DataFrames en uno solo
combined_df = pd.concat(all_prices, ignore_index=True)

# Guardar como CSV unificado
combined_df.to_csv('combined_prices_365_days.csv', index=False, sep=';')

print("✅ Archivo combinado guardado: combined_prices_365_days.csv")


Descargando datos de PEPE...
Descargando datos de DOGE...
Descargando datos de SHIBA...
✅ Archivo combinado guardado: combined_prices_365_days.csv


In [3]:
import requests
import pandas as pd

# Diccionario con los IDs de CoinGecko
coins = {
    'pepe': 'pepe',
    'doge': 'dogecoin',
    'shiba': 'shiba-inu'
}

# Lista para almacenar los DataFrames de cada moneda
all_volatility = []

# Loop para cada moneda
for name, coingecko_id in coins.items():
    print(f"Procesando {name.upper()}...")

    url = f'https://api.coingecko.com/api/v3/coins/{coingecko_id}/ohlc'
    params = {
        'vs_currency': 'usd',
        'days': '365'
    }

    response = requests.get(url, params=params)
    data = response.json()

    if not data:
        print(f"❌ No se pudieron obtener datos para {name.upper()}.")
        continue

    # Crear DataFrame
    df = pd.DataFrame(data, columns=['timestamp', 'open', 'high', 'low', 'close'])

    # Convertir timestamp a fecha
    df['date'] = pd.to_datetime(df['timestamp'], unit='ms').dt.date

    # Calcular métricas de volatilidad
    df['range'] = df['high'] - df['low']
    df['pct_change'] = (df['close'] - df['open']) / df['open'] * 100
    df['rel_volatility'] = (df['high'] - df['low']) / ((df['high'] + df['low']) / 2)

    # Redondear para evitar problemas de formato
    df['range'] = df['range'].round(10)
    df['pct_change'] = df['pct_change'].round(6)
    df['rel_volatility'] = df['rel_volatility'].round(6)

    # Añadir columna con el nombre de la moneda
    df['coin'] = name

    # Seleccionar columnas relevantes
    final_df = df[['date', 'open', 'high', 'low', 'close', 'range', 'pct_change', 'rel_volatility', 'coin']]

    # Añadir al conjunto final
    all_volatility.append(final_df)

# Concatenar todos los DataFrames
combined_volatility = pd.concat(all_volatility, ignore_index=True)

# Guardar como CSV unificado
combined_volatility.to_csv('combined_volatility_365_days.csv', index=False, sep=';', decimal=',')

print("✅ Archivo combinado guardado: combined_volatility_365_days.csv")


Procesando PEPE...
Procesando DOGE...
Procesando SHIBA...
✅ Archivo combinado guardado: combined_volatility_365_days.csv
